# Initial population structure and growth limits

Every run starts from an **initial population**: the plants standing at the start week, one row per
(chain, age) -- vegetative (pre-flowering) plants by weeks since they were planted or selected as
followers, bunched (post-flowering) plants by weeks since bagging, with their ribbon `tag`. There are
three ways to obtain one, in order of preference:

1. **A field census** of an established farm -- or, for a **new plantation**, its **planting
   schedule**: every planting week is a cohort born that week.
2. **Reconstruction from production records** when neither exists: bunches still on the plant are
   counted from the ribbon records, vegetative plants are inferred through the model's own flowering
   law (`reconstruct_initial_population` in py-simba-pop-tools).
3. **Spin-up** when there are no records at all: a steady weekly follower selection sized to the
   farm's number of production units (mats), run to equilibrium (`spin_up_initial_population`).

Grounding: Tixier et al. (2004) calibrate the flowering threshold FT and the sucker-appearance
threshold BT as heat units accumulated **from planting**, so a planting week is a cohort birth week
(thermal age 0). Guarin & Ochoa (2012) start their Urabá simulations from the plantation's
establishment ("la simulación debe partir desde el establecimiento de la plantación") and keep the
number of production units constant throughout -- which the default `rate_basis = "cohort"` does by
itself.

Mode A below means an engine built from the initial population alone (no ribbon or desuckering
records: the model's own flowering and follower selection); Mode B adds the ribbon bagging records,
Mode C the desuckering records too. Datasets: `data/synthetic_new_plantation` (a 170-ha farm planted
from scratch; override with `SIMBA_POP_NEW_PLANTATION_DIR`) and `data/synthetic` (an established
165-ha farm with a field census; override with `SIMBA_POP_DATA_DIR`).

In [ ]:
from pathlib import Path

# This notebook lives in docs/examples/: run it from there so data/ resolves
# (py_simba_pop itself comes from `pip install py-simba-pop`).

In [2]:
import os

import numpy as np
import pandas as pd
import plotly.graph_objects as go

# Reconstruction, spin-up and the cited desuckering practice come from py-simba-pop-tools.
from py_simba_pop_tools.calibration import GUARIN_OCHOA_2012_SUCKER_SELECTION as FARM_PRACTICE
from py_simba_pop_tools.data import (
    generation_interval_weeks,
    reconstruct_initial_population,
    spin_up_initial_population,
)

from py_simba_pop import (
    ChainKind,
    CohortChain,
    CohortChainError,
    ParameterSet,
    PopulationInventoryReader,
    RibbonProductionData,
    RibbonProductionReader,
    SimbaPopEngine,
    SuckerSelectionData,
    SuckerSelectionReader,
    WeatherProvider,
)

NEW_DIR = Path(os.environ.get("SIMBA_POP_NEW_PLANTATION_DIR", "data/synthetic_new_plantation"))
ESTABLISHED_DIR = Path(os.environ.get("SIMBA_POP_DATA_DIR", "data/synthetic"))
# The synthetic established farm: 165 ha at 1,650 production units (mats) per hectare.
ESTABLISHED_HECTARES, PLANTS_PER_HA = 165.0, 1650
print(f"NEW_DIR = {NEW_DIR.as_posix()}; ESTABLISHED_DIR = {ESTABLISHED_DIR.as_posix()}")

params = ParameterSet.from_tixier_defaults()
print(f"rate_basis = {params.general['rate_basis']!r}")


def print_notices(notices):
    for n in notices:
        when = "" if n.week_index is None else f" (week_index {n.week_index})"
        print(f"[{n.kind.value}] {n.subject}{when}: {n.message}")

NEW_DIR = data/synthetic_new_plantation; ESTABLISHED_DIR = data/synthetic
rate_basis = 'cohort'


## 1. Establishing a plantation: staggered planting vs one synchronized planting

A new farm is never planted as one synchronized block: to keep within the packing plant's capacity
and to harvest fruit every week, the agronomist spaces planting over several weeks, planting a
fraction of the hectares each week. `data/synthetic_new_plantation` is a 170-ha farm planted that way,
about 10 ha a week over 17 weeks from 2018-12-31. Its `initial_population.csv`, built from the
farm's planting schedule, holds one pre-flowering cohort per planting week,
`age_weeks` = weeks since planting at `snapshot_date`, the start of the week after the last planting.

In [3]:
weather_new = WeatherProvider.from_file_and_params(NEW_DIR / "weather.csv", params)
print(
    f"{NEW_DIR.name}: weather covers week_index {weather_new.min_week_index}..{weather_new.max_week_index} "
    f"({weather_new.calendar_date_for_week(weather_new.min_week_index)} .. "
    f"{weather_new.calendar_date_for_week(weather_new.max_week_index)})"
)

synthetic_new_plantation: weather covers week_index 0..363 (2018-12-31 .. 2025-12-15)


In [4]:
planting = PopulationInventoryReader.read(NEW_DIR / "initial_population.csv").data
snapshot_date = pd.to_datetime(pd.read_csv(NEW_DIR / "initial_population.csv")["snapshot_date"].iloc[0]).date()
NEW_START_WEEK_INDEX = weather_new.week_index_for_date(snapshot_date)
PLANTED = float(planting["population"].sum())
FIRST_PLANTING_WEEK = NEW_START_WEEK_INDEX - int(planting["age_weeks"].max())

display(planting.assign(planting_week_index=NEW_START_WEEK_INDEX - planting["age_weeks"]).head())
print(
    f"snapshot_date {snapshot_date} -> NEW_START_WEEK_INDEX = {NEW_START_WEEK_INDEX}; {len(planting)} planting cohorts, "
    f"{PLANTED:,.0f} plants, planted in week_index {FIRST_PLANTING_WEEK}..{NEW_START_WEEK_INDEX - int(planting['age_weeks'].min())}"
)

production_result = RibbonProductionReader.read(NEW_DIR / "production.csv")
print_notices(production_result.notices)
production_new = production_result.data
# The plantation's recorded harvest: bunches cut per date, all ribbon colours summed
recorded_cut = production_new.groupby("date")["cut"].sum(min_count=1).dropna()
recorded_cut.index = recorded_cut.index.map(weather_new.week_index_for_date)
NEW_LAST_USABLE_WEEK = min(weather_new.max_week_index, int(production_new["date"].map(weather_new.week_index_for_date).max()))
NEW_N_WEEKS = NEW_LAST_USABLE_WEEK - NEW_START_WEEK_INDEX + 1
recorded_cut = recorded_cut[recorded_cut.index <= NEW_LAST_USABLE_WEEK]
print(f"Simulation window: week_index {NEW_START_WEEK_INDEX}..{NEW_LAST_USABLE_WEEK} ({NEW_N_WEEKS} weeks)")

,chain,population,age_weeks,tag,planting_week_index
0,pre_flowering,17523.0,1,NaN,16
1,pre_flowering,13695.0,2,NaN,15
2,pre_flowering,14454.0,3,NaN,14
3,pre_flowering,16104.0,4,NaN,13
4,pre_flowering,14669.0,5,NaN,12


snapshot_date 2019-04-29 -> NEW_START_WEEK_INDEX = 17; 17 planting cohorts, 280,500 plants, planted in week_index 0..16
[feature_disabled] processed: 'processed' column not present in WindowsPath('data/synthetic_new_plantation/production.csv'); this is a non-paper extension with no Tixier equation to estimate it from, so it will simply not be modeled (treated as absent, not zero).
Simulation window: week_index 17..360 (344 weeks)


The comparison: the same plants planted as **one synchronized cohort** in the first planting week
(`age_weeks = NEW_START_WEEK_INDEX - FIRST_PLANTING_WEEK`), started on the same week with the same
weather. Both runs are Mode A (no field records), so everything after the start week is the model's
own flowering, harvest and follower selection.

In [5]:
synchronized = pd.DataFrame(
    {"chain": ["pre_flowering"], "population": [PLANTED], "age_weeks": [NEW_START_WEEK_INDEX - FIRST_PLANTING_WEEK]}
)
display(synchronized)


def run_mode_a(parameters, seed, n_weeks):
    engine = SimbaPopEngine(parameters, weather_new, seed, start_week_index=NEW_START_WEEK_INDEX)
    return engine, engine.run(n_weeks)


staggered_engine, staggered_out = run_mode_a(params, planting, NEW_N_WEEKS)
_, synchronized_out = run_mode_a(params, synchronized, NEW_N_WEEKS)
print("Construction-time notices of a Mode A engine:")
print_notices(staggered_engine.notices)

,chain,population,age_weeks
0,pre_flowering,280500.0,17


Construction-time notices of a Mode A engine:
[fallback_estimate] ribbon_production: no ribbon-tag production data provided -- every week's flowering output will route to the reserved '__unassigned__' chain, and harvest follows the Tixier hazard curve exactly as in a fully mechanistic run. Provide ribbon_production= to drive specific tags from real bagged counts.
[fallback_estimate] sucker_selection: no sucker-selection data provided -- the pre-flowering chain's weekly inflow will use the model's own mechanistic Eq 3 pooled-sucker prediction for the entire run. Provide sucker_selection= to use real observed desuckering counts instead.


In [6]:
# The packing plant's average weekly intake: the plantation's mean recorded cut after its first harvest
cut_series = recorded_cut.reindex(range(int(recorded_cut[recorded_cut >= 1].index.min()), NEW_LAST_USABLE_WEEK + 1), fill_value=0.0)
LOW_HARVEST = 0.1 * cut_series.mean()


def harvest_summary(harvest):
    # harvest: a Series of bunches per week, indexed by week_index
    first = int(harvest[harvest >= 1.0].index.min())
    after = harvest[harvest.index >= first]
    return {
        "first harvest (week_index)": first,
        "first harvest (week starting)": weather_new.calendar_date_for_week(first),
        "weeks from first harvest to end": len(after),
        "weeks with no harvest (< 1 bunch)": int((after < 1.0).sum()),
        "weeks below 10% of mean recorded cut": int((after < LOW_HARVEST).sum()),
        "share of weeks below 10%": f"{(after < LOW_HARVEST).mean():.1%}",
        "max weekly harvest": round(after.max()),
    }


runs = {
    "staggered planting (model)": staggered_out.set_index("week_index")["Ht"],
    "synchronized planting (model)": synchronized_out.set_index("week_index")["Ht"],
}
summary = pd.DataFrame({label: harvest_summary(ht) for label, ht in runs.items()})
summary["recorded cut"] = pd.Series(harvest_summary(cut_series))
print(f"Mean recorded weekly cut after the first harvest: {cut_series.mean():,.0f} bunches (10% = {LOW_HARVEST:,.0f})")
display(summary)

for label, ht in runs.items():
    model = ht.reindex(cut_series.index)
    first_cycle = cut_series.index[:30]
    for name, weeks in [("first 30 harvest weeks", first_cycle), ("whole run", cut_series.index)]:
        residual = cut_series[weeks] - model[weeks]
        r2 = 1.0 - (residual**2).sum() / ((cut_series[weeks] - cut_series[weeks].mean()) ** 2).sum()
        print(f"{label}: r2 against the recorded weekly cut over the {name}: {r2:.3f}")

# Highest and lowest weekly harvest in each calendar year (2019: from the first harvest in October)
by_year = pd.DataFrame({**runs, "recorded cut": cut_series}).loc[cut_series.index]
by_year.index = [weather_new.calendar_date_for_week(w).year for w in by_year.index]
display(by_year.groupby(level=0).agg(["max", "min"]).round(0).rename_axis("year"))

Mean recorded weekly cut after the first harvest: 8,597 bunches (10% = 860)


,staggered planting (model),synchronized planting (model),recorded cut
first harvest (week_index),42,41,41
first harvest (week starting),2019-10-21,2019-10-14,2019-10-14
weeks from first harvest to end,319,320,320
weeks with no harvest (< 1 bunch),0,0,0
weeks below 10% of mean recorded cut,2,39,3
share of weeks below 10%,0.6%,12.2%,0.9%
max weekly harvest,24225,32566,18318


staggered planting (model): r2 against the recorded weekly cut over the first 30 harvest weeks: 0.998
staggered planting (model): r2 against the recorded weekly cut over the whole run: -0.758
synchronized planting (model): r2 against the recorded weekly cut over the first 30 harvest weeks: -3.585
synchronized planting (model): r2 against the recorded weekly cut over the whole run: -7.227


staggered planting (model)         synchronized planting (model)         \
                            max     min                           max    min   
year                                                                           
2019                    14662.0     0.0                       32566.0    3.0   
2020                    16910.0  1058.0                       30319.0  187.0   
2021                    23472.0  2165.0                       27288.0  329.0   
2022                    24225.0  1917.0                       28603.0  497.0   
2023                    17087.0  2935.0                       23376.0  399.0   
2024                    22485.0  2567.0                       22917.0  451.0   
2025                    20767.0  3824.0                       22706.0  956.0   

     recorded cut          
              max     min  
year                       
2019      14170.0     1.0  
2020      16385.0  1133.0  
2021      18318.0  2407.0  
2022      16511.0  1889.0  
2023      13823.0  2479.0  
2024      14571.0  2922.0  
2025      17246.0  2616.0

In [7]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=[weather_new.calendar_date_for_week(w) for w in cut_series.index], y=cut_series.values,
                         name="recorded cut", line=dict(color="#2b2b2b", width=2)))
fig.add_trace(go.Scatter(x=staggered_out["date"], y=staggered_out["Ht"].round(1), name="Ht, staggered planting (17 weekly lots)",
                         line=dict(color="#2a78d6", width=2)))
fig.add_trace(go.Scatter(x=synchronized_out["date"], y=synchronized_out["Ht"].round(1), name="Ht, one synchronized planting",
                         line=dict(color="#eb6834", width=2)))
fig.add_hline(y=LOW_HARVEST, line_dash="dot", line_color="#898781", annotation_text="10% of mean recorded cut",
              annotation_position="bottom right")
fig.update_layout(
    title=f"{NEW_DIR.name}: weekly harvest, Mode A from the planting schedule vs one synchronized planting",
    xaxis_title="Date",
    yaxis_title="Bunches per week",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

Both plantings cut their first bunches 41-42 weeks after the first lot went in (October 2019, as the
plantation did), and neither ever has a week with no harvest at all: the flowering and harvest
distributions always leave a trickle. The difference is the load on the packing plant. The
synchronized planting harvests in cycles of sharp peaks (32,566 bunches in its first peak week) and
deep troughs (down to 187 bunches a week in 2020), with 39 weeks (12.2%) below 10% of the average
recorded intake, and its yearly peaks are still ~22,700 bunches in 2025. The staggered schedule
reproduces the plantation's recorded first cycle almost exactly (r2 = 0.998 over the first 30 harvest
weeks) and has only 2 such weeks (the record: 3). Later its cycles drift from the record (whole-run
r2 = -0.76; yearly peaks up to 24,225 against at most 18,318 recorded) because Mode A selects
followers on Tixier's calibrated curve, while this farm's desuckering practice is later and broader
(section 2a); with the farm's desuckering records (Mode C) the engine follows the farm instead -- see
[03_validating_farm_simulation.ipynb](03_validating_farm_simulation.ipynb).

### The planted plants alone (follower selection switched off)

With `sucker_selection.amplitude = 0` no followers are selected, so `Ht` is the harvest of the
planted plants only. Under the cohort rate basis every one of them flowers once and is harvested
once, so the run's total harvest must equal the number of plants planted -- the only leak is
`population_epsilon` pruning of sub-0.5-plant tail cohorts, reported week by week in
`pruned_population_this_week`.

In [8]:
no_followers = params.override("sucker_selection", {"amplitude": 0.0})
first_cycle_rows = {}
first_cycle_out = {}
for label, seed in [("staggered planting", planting), ("synchronized planting", synchronized)]:
    _, out = run_mode_a(no_followers, seed, NEW_N_WEEKS)
    first_cycle_out[label] = out
    ht = out.set_index("week_index")["Ht"]
    cumulative = ht.cumsum() / ht.sum()
    peak_week = int(ht.idxmax())
    harvested, pruned, left = ht.sum(), out["pruned_population_this_week"].sum(), out["Nt"].iloc[-1]
    assert abs(harvested + pruned + left - PLANTED) < 1e-6 * PLANTED
    first_cycle_rows[label] = {
        "first harvest (week_index)": int(ht[ht >= 1.0].index.min()),
        "peak weekly harvest": round(ht.max()),
        "peak, % of plants planted": f"{100 * ht.max() / PLANTED:.2f}%",
        "peak week_index": peak_week,
        "weeks from 10% to 90% harvested": int((cumulative >= 0.9).idxmax() - (cumulative >= 0.1).idxmax()),
        "total harvested": round(harvested, 1),
        "pruned (tail cohorts)": round(pruned, 1),
        "still standing at the end": round(left, 1),
        "plants planted": PLANTED,
    }
display(pd.DataFrame(first_cycle_rows))
for label, row in first_cycle_rows.items():
    print(f"{label}: harvested {row['total harvested']:,.1f} + pruned {row['pruned (tail cohorts)']:,.1f} = "
          f"{row['total harvested'] + row['pruned (tail cohorts)']:,.1f} of {PLANTED:,.0f} planted -- every plant harvested once")

,staggered planting,synchronized planting
first harvest (week_index),42,41
peak weekly harvest,15578,32566
"peak, % of plants planted",5.55%,11.61%
peak week_index,55,47
weeks from 10% to 90% harvested,17,10
total harvested,280489.1,280494.0
pruned (tail cohorts),10.9,6.0
still standing at the end,0.0,0.0
plants planted,280500.0,280500.0


staggered planting: harvested 280,489.1 + pruned 10.9 = 280,500.0 of 280,500 planted -- every plant harvested once
synchronized planting: harvested 280,494.0 + pruned 6.0 = 280,500.0 of 280,500 planted -- every plant harvested once


In [9]:
fig = go.Figure()
for label, colour in [("staggered planting", "#2a78d6"), ("synchronized planting", "#eb6834")]:
    out = first_cycle_out[label]
    window = out[out["Ht"] >= 1.0]
    fig.add_trace(go.Scatter(x=window["date"], y=(100 * window["Ht"] / PLANTED).round(3), name=label,
                             line=dict(color=colour, width=2)))
fig.update_layout(
    title=f"{NEW_DIR.name}: harvest of the planted plants alone (follower selection off)",
    xaxis_title="Date",
    yaxis_title="Harvested per week (% of plants planted)",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

Each run harvests exactly the plants planted: 280,489.1 (staggered) and 280,494.0 (synchronized)
bunches, plus 10.9 and 6.0 plants pruned as sub-0.5-plant tails, = 280,500. The synchronized planting
concentrates its first cycle -- 11.61% of all plants harvested in the peak week and 10% to 90% of the
harvest within 10 weeks -- while the 17 weekly lots spread it to a 5.55% peak over 17 weeks: the same
flowering and harvest distributions, summed over 17 week-shifted birth weeks.

### The coverage rule: `start_week_index >= max(age_weeks)`

`start_week_index` is a week in the weather file's own numbering, not a calendar week. A cohort's
`age_weeks` fixes its birth week (`start_week_index - age_weeks`), and the engine replays each
cohort's thermal history from that week, so the birth week must lie inside the weather record. A
start week that is too small implies plantings before the weather data begins:

In [10]:
MAX_AGE = int(planting["age_weeks"].max())
too_early = weather_new.min_week_index + MAX_AGE - 2
try:
    SimbaPopEngine(params, weather_new, planting, start_week_index=too_early)
except CohortChainError as exc:
    print(f"start_week_index={too_early}: {type(exc).__name__}: {exc}")

safe = weather_new.min_week_index + MAX_AGE
SimbaPopEngine(params, weather_new, planting, start_week_index=safe)
print(f"\nstart_week_index={safe} (weather starts at week_index {weather_new.min_week_index}, max(age_weeks) = {MAX_AGE}): "
      f"constructed without error. The snapshot_date gives NEW_START_WEEK_INDEX = {NEW_START_WEEK_INDEX}.")

start_week_index=15: CohortChainError: seed row with age_weeks=17 for chain=pre_flowering implies birth_week_index=-2; reconstructing its thermal history needs weather data back to week_index -1, before the weather data's coverage starts at week_index 0. Raise start_week_index to at least 16, or construct WeatherProvider with weather data and an explicit run_start_date extending back far enough.

start_week_index=17 (weather starts at week_index 0, max(age_weeks) = 17): constructed without error. The snapshot_date gives NEW_START_WEEK_INDEX = 17.


The error names the exact minimum (one week below the rule, because a cohort's own birth week needs no
replay); `start_week_index >= weather.min_week_index + max(age_weeks)` is the always-safe rule. Reading
the start week from `snapshot_date`, as above, satisfies it whenever the weather record starts at or
before the first planting.

## 2. Deriving an initial population

Without a census or a planting schedule the initial population has to be derived.

### 2a. No records at all: spin-up

`spin_up_initial_population(weather, params, mats)` selects `mats / generation interval` followers
every week for 104 weeks from the start of the weather record, then snapshots the engine: the
steady-state age structure of a farm with that many production units. Bunched plants have no known
ribbon colour, so they are left in the reserved `__unassigned__` tag. Here: the established farm's
weather and the synthetic farm's size (165 ha x 1,650 mats/ha).

In [11]:
weather_est = WeatherProvider.from_file_and_params(ESTABLISHED_DIR / "weather.csv", params)
MATS = ESTABLISHED_HECTARES * PLANTS_PER_HA
interval = generation_interval_weeks(weather_est, params)
spun_up = spin_up_initial_population(weather_est, params, MATS)

display(
    spun_up.groupby("chain").agg(
        cohorts=("population", "size"),
        plants=("population", "sum"),
        min_age_weeks=("age_weeks", "min"),
        max_age_weeks=("age_weeks", "max"),
    ).round(0)
)
print(f"Generation interval (selection to follower selection): {interval:.1f} weeks -> {MATS / interval:,.0f} followers selected per week for {MATS:,.0f} mats")
print(f"Standing plants: {spun_up['population'].sum():,.0f} = {spun_up['population'].sum() / MATS:.2f} per mat "
      f"(snapshot_date {spun_up['snapshot_date'].iloc[0]})")

,cohorts,plants,min_age_weeks,max_age_weeks
chain,,,,
post_flowering,15,110875.0,1,15
pre_flowering,78,334206.0,1,78


Generation interval (selection to follower selection): 30.0 weeks -> 9,069 followers selected per week for 272,250 mats
Standing plants: 445,082 = 1.63 per mat (snapshot_date 2018-01-01)


In [12]:
fig = go.Figure()
for chain, colour in [("pre_flowering", "#2a78d6"), ("post_flowering", "#eb6834")]:
    by_age = spun_up[spun_up["chain"] == chain].groupby("age_weeks")["population"].sum()
    fig.add_trace(go.Bar(x=by_age.index, y=by_age.values.round(0), name=chain, marker_color=colour))
fig.update_layout(
    barmode="group",
    title=f"Spin-up: standing plants by age for {MATS:,.0f} mats (Tixier follower-selection curve)",
    xaxis_title="age_weeks",
    yaxis_title="Plants",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

In [13]:
# The spin-up assumes the parameter set's follower-selection curve. The synthetic established farm's
# desuckering practice (Guarin & Ochoa 2012, Table 2) selects followers later and over more weeks:
practice = params.override("sucker_selection", FARM_PRACTICE)
spun_up_practice = spin_up_initial_population(weather_est, practice, MATS)
print(f"Generation interval with the farm's practice: {generation_interval_weeks(weather_est, practice):.1f} weeks; "
      f"spin-up: {spun_up_practice['population'].sum() / MATS:.2f} standing plants per mat")
if ESTABLISHED_DIR.resolve() == Path("data/synthetic").resolve():
    census_total = pd.read_csv(ESTABLISHED_DIR / "initial_population.csv")["population"].sum()
    print(f"Field census of the synthetic farm: {census_total:,.0f} plants = {census_total / MATS:.2f} per mat")

Generation interval with the farm's practice: 31.9 weeks; spin-up: 1.54 standing plants per mat
Field census of the synthetic farm: 417,296 plants = 1.53 per mat


With no records, the spin-up gives 445,082 standing plants for 272,250 mats -- 1.63 per mat, each
mat's mother plant plus, for part of the cycle, her follower -- with vegetative plants aged 1-78 weeks
and bunches 1-15 weeks after bagging. The structure follows the parameter set's follower-selection
curve: with the synthetic farm's own desuckering practice the generation interval is 31.9 instead of
30.0 weeks and the spin-up gives 1.54 plants per mat, close to the 1.53 of its field census. A spin-up
is a sound start when nothing is recorded, as long as its selection timing reflects the farm's
practice.

### 2b. Production records only: reconstruction

For a farm with ribbon records but no census, `reconstruct_initial_population(production_df,
weather_df, params, snapshot_week_index)` counts the bunches still on the plant per colour from the
bagging / cut / loss ledger, and infers the vegetative plants by fitting weekly follower selections
whose model flowering reproduces the recorded bagging -- using only records before the snapshot week.
We check it where the truth is known: on the established farm, reconstruct at the week two years after
the census and compare with the state of a Mode C engine (census + bagging + desuckering records)
stepped from the census week to that week.

In [14]:
census = PopulationInventoryReader.read(ESTABLISHED_DIR / "initial_population.csv").data
census_date = pd.to_datetime(pd.read_csv(ESTABLISHED_DIR / "initial_population.csv")["snapshot_date"].iloc[0]).date()
START_WEEK_INDEX = weather_est.week_index_for_date(census_date)
RECON_WEEK_INDEX = START_WEEK_INDEX + 104

reconstructed, recon_report = reconstruct_initial_population(
    pd.read_csv(ESTABLISHED_DIR / "production.csv"), pd.read_csv(ESTABLISHED_DIR / "weather.csv"), params, RECON_WEEK_INDEX
)
report = recon_report
print(f"Reconstruction at the start of week_index {report['snapshot_week_index']} ({report['snapshot_date']}), from records before it:")
print(f"  bagging fit R2: weekly {report['bagging_fit_r2_weekly']:.4f}, 8-week totals {report['bagging_fit_r2_8_week']:.4f}")
print(f"  followers selected per bagged bunch: {report['selections_per_bagged_bunch']:.3f} (one follower per mother gives ~1)")
print(f"  last selection week identified by bagging: week_index {report['last_identified_selection_week']} "
      f"(vegetative plants younger than {report['snapshot_week_index'] - report['last_identified_selection_week']} weeks come from the model's own selection rule)")
print(f"  vegetative plants: {report['pre_flowering_plants']:,.0f} (smoothing sensitivity {report['smoothing_range_pre_total'][0]:,.0f}"
      f"..{report['smoothing_range_pre_total'][1]:,.0f})")
print(f"  bunches on the plant: {report['standing_bunches']:,.0f} (later cuts traced back to them: {report['later_cuts_of_standing']:,.0f}; "
      f"cuts not matched to a standing cohort: {report['unmatched_cut']:,.0f})")
print(f"  generation interval {report['generation_interval_weeks']:.1f} weeks -> implied production units ~{report['implied_mats']:,.0f} mats")

Reconstruction at the start of week_index 260 (2020-12-28), from records before it:
  bagging fit R2: weekly 0.9978, 8-week totals 0.9999
  followers selected per bagged bunch: 0.992 (one follower per mother gives ~1)
  last selection week identified by bagging: week_index 219 (vegetative plants younger than 41 weeks come from the model's own selection rule)
  vegetative plants: 324,233 (smoothing sensitivity 324,082..324,402)
  bunches on the plant: 109,719 (later cuts traced back to them: 108,130; cuts not matched to a standing cohort: 803)
  generation interval 30.0 weeks -> implied production units ~262,417 mats


In [15]:
production_est = RibbonProductionReader.read(ESTABLISHED_DIR / "production.csv")
print_notices(production_est.notices)
ribbon_est = RibbonProductionData(production_est.data, weather_est)

# sucker_selection.csv is optional: without it the reference run is Mode B (the model selects the followers)
if (ESTABLISHED_DIR / "sucker_selection.csv").exists():
    sucker_result = SuckerSelectionReader.read(ESTABLISHED_DIR / "sucker_selection.csv")
    print_notices(sucker_result.notices)
    sucker_selection = SuckerSelectionData(sucker_result.data, weather_est)
    REFERENCE = "Mode C"
else:
    sucker_selection = None
    REFERENCE = "Mode B"
    print(f"No sucker_selection.csv in {ESTABLISHED_DIR.as_posix()}: the reference run is Mode B.")

reference = SimbaPopEngine(
    params, weather_est, census, start_week_index=START_WEEK_INDEX,
    ribbon_production=ribbon_est, sucker_selection=sucker_selection, tags=list(ribbon_est.tags()),
)
reference.run(RECON_WEEK_INDEX - START_WEEK_INDEX)
reference_state = reference.snapshot_cohorts()
reference_state = reference_state[reference_state["population"] >= 0.5]
weekly_notices = [n for n in reference.notices if n.week_index is not None]
print(f"{REFERENCE} engine: census week_index {START_WEEK_INDEX} stepped to {reference.current_week_index}; "
      f"{len(weekly_notices)} weekly notices on the way")
print_notices(weekly_notices[:5])

label_recon = "reconstructed (production records only)"
label_ref = f"{REFERENCE} engine state"
totals = pd.DataFrame({
    label_recon: reconstructed.groupby("chain")["population"].sum(),
    label_ref: reference_state.groupby("chain")["population"].sum(),
})
totals.loc["total"] = totals.sum()
totals["difference"] = (totals[label_recon] / totals[label_ref] - 1).map("{:+.1%}".format)
display(totals.round(0))

bunches = pd.DataFrame({
    label_recon: reconstructed[reconstructed["chain"] == "post_flowering"].groupby("tag")["population"].sum(),
    label_ref: reference_state[reference_state["chain"] == "post_flowering"].groupby("tag")["population"].sum(),
}).fillna(0.0)
bunches["difference"] = (bunches[label_recon] / bunches[label_ref] - 1).map("{:+.1%}".format)
display(bunches.round(0).rename_axis("bunches on the plant, by ribbon colour").T)

[feature_disabled] processed: 'processed' column not present in WindowsPath('data/synthetic/production.csv'); this is a non-paper extension with no Tixier equation to estimate it from, so it will simply not be modeled (treated as absent, not zero).
Mode C engine: census week_index 156 stepped to 260; 0 weekly notices on the way


,reconstructed (production records only),Mode C engine state,difference
chain,,,
post_flowering,109719.0,109161.0,+0.5%
pre_flowering,324233.0,321181.0,+1.0%
total,433952.0,430342.0,+0.8%


"bunches on the plant, by ribbon colour",Amarillo,Azul,Blanco,Café,Morado,Negro,Rojo,Verde
reconstructed (production records only),16578.0,16945.0,16756.0,8683.0,16551.0,10304.0,8548.0,15354.0
Mode C engine state,16513.0,16880.0,16695.0,8611.0,16475.0,10194.0,8484.0,15310.0
difference,+0.4%,+0.4%,+0.4%,+0.8%,+0.5%,+1.1%,+0.8%,+0.3%


In [16]:
YOUNG = report["snapshot_week_index"] - report["last_identified_selection_week"]
veg = pd.DataFrame({
    label_recon: reconstructed[reconstructed["chain"] == "pre_flowering"].groupby("age_weeks")["population"].sum(),
    label_ref: reference_state[reference_state["chain"] == "pre_flowering"].groupby("age_weeks")["population"].sum(),
}).fillna(0.0)

fig = go.Figure()
fig.add_trace(go.Scatter(x=veg.index, y=veg[label_ref].round(0), name=label_ref, mode="lines+markers",
                         line=dict(color="#2b2b2b", width=2), marker=dict(size=4)))
fig.add_trace(go.Scatter(x=veg.index, y=veg[label_recon].round(0), name=label_recon, mode="lines+markers",
                         line=dict(color="#2a78d6", width=2, dash="dot"), marker=dict(size=4)))
fig.add_vline(x=YOUNG - 0.5, line_dash="dot", line_color="#898781",
              annotation_text="younger: model's own selection timing", annotation_position="top left")
fig.update_layout(
    title=f"{ESTABLISHED_DIR.name}: vegetative plants by age at week_index {RECON_WEEK_INDEX}",
    xaxis_title="age_weeks (weeks since selection)",
    yaxis_title="Plants",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

print(f"{veg.loc[veg.index < YOUNG, label_recon].sum() / veg[label_recon].sum():.1%} of the reconstructed vegetative plants "
      f"are younger than {YOUNG} weeks")
for name, rows in [(f"younger than {YOUNG} weeks", veg.index < YOUNG), (f"{YOUNG} weeks and older", veg.index >= YOUNG)]:
    part = veg[rows].sum()
    print(f"Vegetative plants {name}: reconstructed {part[label_recon]:,.0f}, {REFERENCE} {part[label_ref]:,.0f} "
          f"({part[label_recon] / part[label_ref] - 1:+.1%}); mean absolute difference per age "
          f"{(veg[rows][label_recon] - veg[rows][label_ref]).abs().mean():,.0f} plants")

98.0% of the reconstructed vegetative plants are younger than 41 weeks
Vegetative plants younger than 41 weeks: reconstructed 317,869, Mode C 314,917 (+0.9%); mean absolute difference per age 907 plants
Vegetative plants 41 weeks and older: reconstructed 6,364, Mode C 6,264 (+1.6%); mean absolute difference per age 7 plants


From production records alone the reconstruction lands within about 1% of the Mode C engine's state:
+1.0% vegetative plants, +0.5% bunches, +0.3% to +1.1% per ribbon colour (the bunch ledger is
observed; the weekly bagging is fitted with R2 = 0.998). 98% of the vegetative plants are younger than
41 weeks: they were selected after week_index 219, the last selection week the bagging record
identifies, and have not flowered yet, so the reconstruction must assume the model's default (Tixier)
selection timing for them. Their number is right -- one follower per identified mother, +0.9% -- but
their ages follow Tixier's curve rather than the farm's later, broader practice, hence the per-age
differences in the chart (~900 plants per age class on average; 7 for the older plants).

## 3. Pre-/post-flowering visibility

Every `EngineWeeklyRecord` carries `Nt_pre` (vegetative plants) and `Nt_post` (bunches on the plant,
all tags), with `Nt = Nt_pre + Nt_post`; the age structure behind them comes from
`engine.snapshot_cohorts()`, in the same schema as the census. Below: a Mode A run of the established
farm from its census -- no bagging or desuckering records, so flowering, harvest and follower
selection are all the model's own -- stepped one week at a time with a snapshot at the start of every
week, as in notebook 01.

In [17]:
LAST_USABLE_WEEK = min(weather_est.max_week_index, int(production_est.data["date"].map(weather_est.week_index_for_date).max()))
N_WEEKS = LAST_USABLE_WEEK - START_WEEK_INDEX + 1
SNAPSHOT_WEEKS = list(range(START_WEEK_INDEX, START_WEEK_INDEX + N_WEEKS))
SNAPSHOT_DATES = [weather_est.calendar_date_for_week(w) for w in SNAPSHOT_WEEKS]


def run_with_snapshots(engine, n_weeks):
    # Steps `engine` n_weeks times, snapshotting the cohorts at the start of every week
    snapshots = []
    for _ in range(n_weeks):
        snapshot = engine.snapshot_cohorts()
        snapshots.append(snapshot[["chain", "age_weeks", "population"]].assign(week_index=engine.current_week_index))
        engine.step()
    return engine.get_output(), pd.concat(snapshots, ignore_index=True)


mode_a = SimbaPopEngine(params, weather_est, census, start_week_index=START_WEEK_INDEX)
mode_a_output, mode_a_history = run_with_snapshots(mode_a, N_WEEKS)
assert (mode_a_output["Nt_pre"] + mode_a_output["Nt_post"] - mode_a_output["Nt"]).abs().max() < 1e-6
print(f"Mode A from the census: week_index {START_WEEK_INDEX}..{LAST_USABLE_WEEK} ({N_WEEKS} weeks); Nt_pre + Nt_post == Nt every week")
for column in ["Nt", "Nt_pre", "Nt_post"]:
    series = mode_a_output[column]
    print(f"  {column}: {series.min():,.0f} .. {series.max():,.0f} (start {series.iloc[0]:,.0f}, end {series.iloc[-1]:,.0f})")
print(f"  followers seeded {mode_a_output['sucker_input_used'].sum():,.0f}, plants flowered {mode_a_output['FLt'].sum():,.0f}, "
      f"bunches harvested {mode_a_output['Ht'].sum():,.0f}")

fig = go.Figure()
fig.add_trace(go.Scatter(x=mode_a_output["date"], y=mode_a_output["Nt_pre"].round(0), name="Nt_pre (pre-flowering)",
                         mode="lines", stackgroup="one", line=dict(color="#2a78d6", width=0.5), fillcolor="#2a78d6"))
fig.add_trace(go.Scatter(x=mode_a_output["date"], y=mode_a_output["Nt_post"].round(0), name="Nt_post (post-flowering)",
                         mode="lines", stackgroup="one", line=dict(color="#eb6834", width=0.5), fillcolor="#eb6834"))
fig.update_layout(
    title=f"{ESTABLISHED_DIR.name}: standing plants, Mode A from the census",
    xaxis_title="Date",
    yaxis_title="Plants",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

Mode A from the census: week_index 156..516 (361 weeks); Nt_pre + Nt_post == Nt every week
  Nt: 408,279 .. 443,383 (start 419,575, end 428,075)
  Nt_pre: 307,440 .. 334,834 (start 323,861, end 317,879)
  Nt_post: 92,508 .. 122,047 (start 95,715, end 110,196)
  followers seeded 3,089,795, plants flowered 3,094,226, bunches harvested 3,078,850


In [18]:
PRE_BANDS = [(0, 9), (10, 19), (20, 29), (30, 39), (40, None)]  # weeks since selection/planting
POST_BANDS = [(0, 4), (5, 9), (10, 14), (15, None)]  # weeks since bagging
PRE_COLOURS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#0d366b"]  # young -> old
POST_COLOURS = ["#f5b08a", "#eb6834", "#b8461a", "#7a2c0c"]  # just bagged -> oldest


def band_label(lo, hi):
    return f"{lo}+ wk" if hi is None else f"{lo}-{hi} wk"


def population_by_band(history, chain, bands, weeks):
    # Population per (snapshot week, age band) for one chain -- weeks as rows, bands as columns
    rows = history[history["chain"] == chain]
    labels = [band_label(lo, hi) for lo, hi in bands]
    if rows.empty:
        return pd.DataFrame(0.0, index=weeks, columns=labels)
    band = pd.cut(rows["age_weeks"], bins=[lo for lo, _ in bands] + [float("inf")], labels=labels, right=False)
    table = rows.groupby([rows["week_index"], band], observed=False)["population"].sum().unstack(fill_value=0.0)
    return table.reindex(index=weeks, columns=labels, fill_value=0.0)


def band_figure(table, dates, colours, title, yaxis_title, groupnorm=None):
    fig = go.Figure()
    for label, colour in zip(table.columns, colours):
        fig.add_trace(go.Scatter(x=dates, y=table[label].round(0), name=label, mode="lines", stackgroup="bands",
                                 groupnorm=groupnorm, line=dict(color=colour, width=0.5), fillcolor=colour))
    fig.update_layout(title=title, xaxis_title="Date", yaxis_title=yaxis_title, template="plotly_white",
                      legend=dict(orientation="h", y=1.15))
    fig.show()


pre_bands = population_by_band(mode_a_history, "pre_flowering", PRE_BANDS, SNAPSHOT_WEEKS)
post_bands = population_by_band(mode_a_history, "post_flowering", POST_BANDS, SNAPSHOT_WEEKS)
display(
    pd.concat({"pre_flowering": pre_bands.mean(), "post_flowering": post_bands.mean()})
    .rename("mean standing plants per snapshot")
    .round(0)
    .to_frame()
)
band_figure(pre_bands, SNAPSHOT_DATES, PRE_COLOURS, f"{ESTABLISHED_DIR.name}: pre-flowering plants by weeks since selection/planting (Mode A)", "Plants")
band_figure(post_bands, SNAPSHOT_DATES, POST_COLOURS, f"{ESTABLISHED_DIR.name}: bunches on the plant by weeks since bagging (Mode A)", "Bunches")

mean standing plants per snapshot
               age_weeks                                   
pre_flowering  0-9 wk                               77062.0
               10-19 wk                             85475.0
               20-29 wk                             85467.0
               30-39 wk                             61635.0
               40+ wk                                9590.0
post_flowering 0-4 wk                               34291.0
               5-9 wk                               42817.0
               10-14 wk                             30281.0
               15+ wk                                  70.0

From the census, Mode A keeps the established farm between 408,279 and 443,383 standing plants over
361 weeks (2019-2025; `Nt_pre` 307k-335k, `Nt_post` 93k-122k) with no cap: 3,089,795 followers
seeded, 3,094,226 plants flowered and 3,078,850 bunches harvested -- one follower per bunch. The
vegetative plants sit mostly in the 0-39-week bands (the 40+ band averages 9,590 plants, the slow tail
of the flowering distribution), and almost every bunch is cut within 15 weeks of bagging (70 bunches
on average in the 15+ band).

## 4. Growth limits: rate basis

Tixier et al. (2004)'s flowering, sucker-selection and harvest curves (Eqs 17-19) can be read two
ways, set by `general.rate_basis`. Under `"cohort"` (the default; Ochoa 2010's corrected Eq 5, as
implemented by Guarin & Ochoa 2012) each curve is the distribution, over weeks since its threshold,
of the cohort's plants, rescaled to total 1. Under `"current_pool"` (the legacy reading) each rate
multiplies the cohort's current, already-depleted pool. First, one planted cohort followed on its own:

In [19]:
legacy = params.override("general", {"rate_basis": "current_pool"})


def follow_one_cohort(parameters, birth_week, plants=1.0e6):
    # One pre-flowering cohort with no pruning below the max-age cap: its lifetime flowering and followers
    chain = CohortChain(ChainKind.PRE_FLOWERING, parameters.override("general", {"population_epsilon": 0.0}))
    chain.add_new_cohort(plants, birth_week)
    flowered = followers = 0.0
    for week in range(birth_week + 1, weather_new.max_week_index + 1):
        rates = chain.calc_rates(weather_new.thermal_increment(week))
        flowered += rates.sum_chain_exit
        followers += rates.sum_sucker + rates.sum_sucker_parous
        chain.integrate(rates, week)
        if chain.total_population() <= 0.0:
            break
    return {
        "share that ever flowers": f"{flowered / plants:.3f}",
        "share never flowering (removed at the max-age cap)": f"{max(0.0, 1 - flowered / plants):.3f}",
        "followers selected per plant": f"{followers / plants:.3f}",
    }


display(pd.DataFrame({
    "cohort (default)": follow_one_cohort(params, FIRST_PLANTING_WEEK),
    "current_pool (legacy)": follow_one_cohort(legacy, FIRST_PLANTING_WEEK),
}))
print(f"max_age_weeks_vegetative = {params.general['max_age_weeks_vegetative']} (the safety cap that finally removes never-flowering plants)")

,cohort (default),current_pool (legacy)
share that ever flowers,1.000,0.649
share never flowering (removed at the max-age cap),0.000,0.351
followers selected per plant,1.000,2.632


max_age_weeks_vegetative = 150 (the safety cap that finally removes never-flowering plants)


Now the whole plantation, from the same staggered planting, in Mode A for six years under (i) the
cohort basis, (ii) the legacy current-pool basis and (iii) the legacy basis with
`sucker_selection.carrying_capacity` set to the number of plants planted:

In [20]:
GROWTH_WEEKS = min(6 * 52, NEW_N_WEEKS)
legacy_capped = legacy.override("sucker_selection", {"carrying_capacity": PLANTED})
scenarios = {
    "(i) cohort (default)": params,
    "(ii) current_pool (legacy)": legacy,
    "(iii) current_pool + carrying_capacity": legacy_capped,
}
GROWTH_SNAPSHOT_WEEKS = list(range(NEW_START_WEEK_INDEX, NEW_START_WEEK_INDEX + GROWTH_WEEKS))
GROWTH_DATES = [weather_new.calendar_date_for_week(w) for w in GROWTH_SNAPSHOT_WEEKS]
growth, growth_history = {}, {}
for label, parameters in scenarios.items():
    engine = SimbaPopEngine(parameters, weather_new, planting, start_week_index=NEW_START_WEEK_INDEX)
    growth[label], growth_history[label] = run_with_snapshots(engine, GROWTH_WEEKS)

rows = {}
for label, out in growth.items():
    history = growth_history[label]
    last_pre = history[(history["week_index"] == history["week_index"].max()) & (history["chain"] == "pre_flowering")]
    row = {"Nt at start": f"{out['Nt'].iloc[0]:,.0f}"}
    for year in range(1, GROWTH_WEEKS // 52 + 1):
        row[f"Nt after year {year}"] = f"{out['Nt'].iloc[52 * year - 1]:,.0f}"
    lowest = int(out["Nt"].idxmin())
    row["Nt min / plants planted"] = f"{out['Nt'].min() / PLANTED:.2f} (week of {out.loc[lowest, 'date']})"
    row["Nt max / plants planted"] = f"{out['Nt'].max() / PLANTED:,.2f}"
    row["followers seeded per plant flowered"] = f"{out['sucker_input_used'].sum() / out['FLt'].sum():.2f}"
    row["plants removed at the max-age cap"] = f"{out['pruned_population_this_week'].sum():,.0f}"
    share_old = last_pre.loc[last_pre["age_weeks"] >= 40, "population"].sum() / last_pre["population"].sum()
    row["share of vegetative plants aged 40+ wk at the end"] = f"{share_old:.1%}"
    first_harvest = int(out["Ht"].ge(1.0).to_numpy().argmax())
    row["weeks with no harvest after the first"] = f"{int((out['Ht'].iloc[first_harvest:] < 1.0).sum())}"
    rows[label] = row
display(pd.DataFrame(rows))
legacy_nt = growth["(ii) current_pool (legacy)"]["Nt"]
print(f"Legacy basis: Nt grows {(legacy_nt.iloc[-1] / legacy_nt.iloc[-53]):.1f}x over the last year, "
      f"{(legacy_nt.iloc[-1] / legacy_nt.iloc[0]) ** (52 / (len(legacy_nt) - 1)):.1f}x per year on average")

,(i) cohort (default),(ii) current_pool (legacy),(iii) current_pool + carrying_capacity
Nt at start,"280,500","280,500","280,500"
Nt after year 1,"442,072","1,873,021","104,974"
Nt after year 2,"385,575","7,170,513","103,334"
Nt after year 3,"501,602","49,325,508","16,883"
Nt after year 4,"500,061","305,634,068","39,356"
Nt after year 5,"426,983","1,496,710,582","136,992"
Nt after year 6,"419,927","6,312,000,794","220,077"
Nt min / plants planted,1.00 (week of 2019-04-29),1.00 (week of 2019-04-29),0.04 (week of 2022-03-14)
Nt max / plants planted,1.95,"22,502.68",1.00
followers seeded per plant flowered,1.01,4.40,1.00


Legacy basis: Nt grows 4.2x over the last year, 5.3x per year on average


In [21]:
GROWTH_COLOURS = {"(i) cohort (default)": "#2a78d6", "(ii) current_pool (legacy)": "#e34948", "(iii) current_pool + carrying_capacity": "#7b3fb5"}
fig = go.Figure()
for label, out in growth.items():
    fig.add_trace(go.Scatter(x=out["date"], y=out["Nt"].round(0), name=label, line=dict(color=GROWTH_COLOURS[label], width=2)))
fig.add_hline(y=PLANTED, line_dash="dot", line_color="#898781", annotation_text="plants planted (= carrying_capacity in iii)",
              annotation_position="bottom right")
fig.update_layout(
    title=f"{NEW_DIR.name}: standing plants Nt from the same staggered planting, by rate basis (Mode A)",
    xaxis_title="Date",
    yaxis_title="Nt (log scale)",
    yaxis_type="log",
    template="plotly_white",
    legend=dict(orientation="h", y=1.15),
)
fig.show()

legacy_bands = population_by_band(growth_history["(ii) current_pool (legacy)"], "pre_flowering", PRE_BANDS, GROWTH_SNAPSHOT_WEEKS)
band_figure(legacy_bands, GROWTH_DATES, PRE_COLOURS,
            f"{NEW_DIR.name}: pre-flowering plants by weeks since selection/planting, current_pool (legacy)",
            "Share of pre-flowering plants (%)", groupnorm="percent")
cohort_bands = population_by_band(growth_history["(i) cohort (default)"], "pre_flowering", PRE_BANDS, GROWTH_SNAPSHOT_WEEKS)
for label, table in [("cohort (default)", cohort_bands), ("current_pool (legacy)", legacy_bands)]:
    share = table["40+ wk"] / table.sum(axis=1)
    print(f"{label}: vegetative plants aged 40+ weeks, share over the last year {share.iloc[-52:].min():.1%} .. {share.iloc[-52:].max():.1%}")

cohort (default): vegetative plants aged 40+ weeks, share over the last year 0.6% .. 5.7%
current_pool (legacy): vegetative plants aged 40+ weeks, share over the last year 9.1% .. 17.0%


Under the cohort basis (i) every plant flowers once and leaves exactly one follower, so the number of
production units stays at the 280,500 mats planted: over six years 1.01 followers are seeded per plant
flowered, and `Nt` stays between 1.00 and 1.95 plants per mat (a mother, plus her follower for part
of the cycle) without any cap. Under the legacy current-pool reading (ii) 35% of every cohort never
flowers while each plant yields 2.6 followers, so `Nt` grows ~5x a year (5.3x on average, to 6.3
billion plants in six years) and the never-flowering plants pile up in the 40+-week band (9.1-17.0% of
the vegetative plants over the last year, against 0.6-5.7% under the cohort basis) until the 150-week
safety cap removes them (20.4 million plants). A `carrying_capacity` (iii) only caps the count: it
throttles follower selection as last week's `Nt` approaches the cap (none at all at or above it), so
the plantation collapses to 4% of the planted count by March 2022, has 6 weeks without harvest, and
ends with 52% of its vegetative plants 40+ weeks old -- the biology underneath is unchanged.

## Takeaways

1. **A new plantation starts from its planting schedule**, one cohort per planting week (Tixier et al.
   2004 count FT and BT from planting). Spacing the planting over 17 weeks reproduces the plantation's
   recorded first harvest cycle (r2 = 0.998) and keeps the packing plant supplied every week; one
   synchronized planting of the same plants gives sharp peaks and 39 weeks below 10% of the average
   intake.
2. **`start_week_index >= weather.min_week_index + max(age_weeks)`**: every cohort's birth week must
   lie within the weather record. Reading the start week from `snapshot_date` satisfies it; a
   violation raises a `CohortChainError` naming the minimum start week.
3. **No census or planting schedule?** `reconstruct_initial_population` rebuilds the standing plants
   from production records alone to within about 1% of the Mode C state -- the ages of the youngest
   vegetative plants follow the model's default selection timing. With no records at all,
   `spin_up_initial_population` gives the steady state for a number of mats, close to the census's
   plants per mat once the farm's desuckering practice is used.
4. **`Nt_pre` / `Nt_post`** give the stage split every week; `snapshot_cohorts()` gives the full age
   structure in the census schema.
5. **The population stays bounded because of the rate basis, not a cap.** Under the default
   `"cohort"` basis every plant flowers once and keeps one follower, so the number of mats is
   constant. The legacy `"current_pool"` reading grows ~5x a year with 35% of plants never flowering,
   and `carrying_capacity` only caps the count (boom and bust); it does not fix the biology.